# Өгөгдлийн цэвэрлэлт

unegui.mn-ээс татсан түүхий өгөгдлийг машин сургалтад тохирсон хэлбэрт оруулна.

**Алхмууд:**
1. Өгөгдлийг ачаалж, ерөнхий байдлыг шалгах
2. Давхардсан зар арилгах
3. Үнийн outlier-уудыг шалгах ба зайлуулах
4. Талбай, өрөөний тоо, оны логик шалгалт
5. Категорийн хувьсагчдыг кодлох
6. Шинэ feature нэмэх (м²-ийн үнэ, барилгын нас)
7. Цэвэр өгөгдлийг хадгалах

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')

RAW_PATH = Path('../data/test.csv')
OUT_PATH = Path('../data/listings_clean.csv')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Ачаалж шалгах

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f'Хэлбэр: {df.shape}')
df.head(3)

In [ ]:
df.info()

In [ ]:
# Дутуу утгуудын хувь
missing = (df.isna().sum() / len(df) * 100).round(1)
missing[missing > 0].sort_values(ascending=False)

## 2. Давхардсан зар арилгах

In [ ]:
before = len(df)
df = df.drop_duplicates(subset='url').reset_index(drop=True)
print(f'URL-ээр давхардсан: {before - len(df)} мөр устгав')

# Title + price + area-аар давхардсан (өөр URL-тай ч ижил зар)
before = len(df)
df = df.drop_duplicates(subset=['title', 'price_mnt', 'area_m2']).reset_index(drop=True)
print(f'Контентоор давхардсан: {before - len(df)} мөр устгав')
print(f'Үлдсэн: {len(df)}')

## 3. Үнийн цэвэрлэлт

Зорилго: түрээсийн зарыг (сар бүр төлдөг бага үнэ) болон алдаатай оруулсан outlier-уудыг хасна.

In [ ]:
# Үнэгүй мөр устгах
df = df[df['price_mnt'].notna()].copy()

# Үнийн тархалт
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['price_mnt'] / 1e6, bins=40)
axes[0].set_xlabel('Үнэ (сая ₮)')
axes[0].set_title('Үнийн тархалт')
axes[1].hist(np.log10(df['price_mnt']), bins=40)
axes[1].set_xlabel('log10(Үнэ)')
axes[1].set_title('Лог-хэлбэр')
plt.tight_layout(); plt.show()

print((df['price_mnt'] / 1e6).describe().round(1))

In [ ]:
# Худалдааны үнийн доод/дээд хязгаар
# Доод: 30 сая ₮ — энэнээс доош ихэнхдээ түрээс эсвэл алдаа
# Дээд: 5 тэрбум ₮ — энэнээс дээш ховор бөгөөд outlier нөлөөлдөг
MIN_PRICE = 30_000_000
MAX_PRICE = 5_000_000_000

before = len(df)
df = df[(df['price_mnt'] >= MIN_PRICE) & (df['price_mnt'] <= MAX_PRICE)].copy()
print(f'Үнийн хязгаараас гадуур: {before - len(df)} мөр устгав → {len(df)} үлдсэн')

## 4. Талбай, өрөө, оны цэвэрлэлт

In [ ]:
# Талбай: 15-500 м² хооронд логик утга
before = len(df)
df = df[(df['area_m2'] >= 15) & (df['area_m2'] <= 500)].copy()
print(f'Талбайн алдаа: {before - len(df)} мөр устгав')

# Барилгын он: 1950-аас одоо хүртэл (төлөвлөгдсөн он 2027 хүртэл)
before = len(df)
df = df[df['year_built'].between(1950, 2030, inclusive='both') | df['year_built'].isna()].copy()
print(f'Оны алдаа: {before - len(df)} мөр устгав')

# Давхар нь нийт давхараас хэтрэх боломжгүй
mask = df['floor'] > df['total_floors']
print(f'Давхар > нийт давхар: {mask.sum()} мөр (зөвлөмж: үлдээж тэмдэглэе)')
df.loc[mask, ['floor', 'total_floors']] = np.nan  # эргэлзээтэй утгыг NaN болгох

print(f'\nЦэвэрлэлтийн дараа: {len(df)} зар')

In [ ]:
# Өрөөний тоо: title-аас задалсан, дутуу мөрийг median-аар нөхөх
df['rooms'] = df['rooms'].fillna(df.groupby(pd.cut(df['area_m2'], bins=[0,40,60,90,200,500]))['rooms'].transform('median'))
df['rooms'] = df['rooms'].fillna(df['rooms'].median())
df['rooms'] = df['rooms'].astype(int)

df['rooms'].value_counts().sort_index()

## 5. Категорийн хувьсагч кодлох

In [ ]:
# Тагттай эсэх — bool болгох
def has_balcony(s):
    if pd.isna(s): return np.nan
    s = str(s).lower()
    if 'байхгүй' in s or 'үгүй' in s: return 0
    return 1

df['has_balcony'] = df['balcony'].apply(has_balcony)
df['has_garage'] = df['garage'].apply(lambda s: 0 if pd.isna(s) or 'байхгүй' in str(s).lower() else 1)
df['has_elevator'] = df['elevator'].apply(lambda s: 1 if pd.notna(s) and 'байгаа' in str(s).lower() else 0)

# Барилгын явц — ашиглалтад орсон vs баригдаж байгаа
df['is_finished'] = df['construction_status'].apply(
    lambda s: 1 if pd.notna(s) and 'ашиглалт' in str(s).lower() else 0
)

# Тагтны тоо — "1 тагттай", "2 тагттай" гэх мэтээс тоо ялгах
df['balcony_count'] = df['balcony'].str.extract(r'(\d+)').astype(float).fillna(0).astype(int)

df[['has_balcony', 'has_garage', 'has_elevator', 'is_finished', 'balcony_count']].describe()

## 6. Шинэ feature нэмэх

In [ ]:
CURRENT_YEAR = 2025

df['price_per_m2'] = df['price_mnt'] / df['area_m2']
df['building_age'] = (CURRENT_YEAR - df['year_built']).clip(lower=0)
df['log_price'] = np.log(df['price_mnt'])
df['floor_ratio'] = df['floor'] / df['total_floors']  # дунд давхар vs дээд давхар

df[['price_mnt', 'area_m2', 'price_per_m2', 'building_age', 'log_price']].describe().round(2)

## 7. Дүүрэг тус бүрийн дундаж м²-ийн үнэ

In [ ]:
district_stats = df.groupby('district').agg(
    n=('price_mnt', 'count'),
    median_price_m2=('price_per_m2', 'median'),
    mean_price_m2=('price_per_m2', 'mean'),
).sort_values('median_price_m2', ascending=False)
district_stats['median_price_m2'] = (district_stats['median_price_m2'] / 1e6).round(2)
district_stats['mean_price_m2'] = (district_stats['mean_price_m2'] / 1e6).round(2)
district_stats.rename(columns={'median_price_m2': 'med_m²(сая₮)', 'mean_price_m2': 'mean_m²(сая₮)'})

In [ ]:
# Цөөн зартай дүүргийг "Бусад" болгох (зөвхөн 30+ зартай дүүргийг үлдээх)
MIN_PER_DISTRICT = 30
counts = df['district'].value_counts()
rare_districts = counts[counts < MIN_PER_DISTRICT].index
df['district_grouped'] = df['district'].where(~df['district'].isin(rare_districts), 'Бусад')
df['district_grouped'].value_counts()

## 8. Эцсийн шалгалт ба хадгалах

In [ ]:
# Загварчлалд хэрэгтэй баганууд
MODEL_COLS = [
    'price_mnt', 'log_price',
    'area_m2', 'rooms', 'floor', 'total_floors', 'floor_ratio',
    'building_age', 'year_built',
    'has_balcony', 'balcony_count', 'has_garage', 'has_elevator', 'is_finished',
    'district', 'district_grouped',
    'window_type', 'door_type', 'floor_material',
    'price_per_m2',
]

df_model = df[MODEL_COLS].copy()
print(f'Эцсийн өгөгдөл: {df_model.shape}')
print(f'\nДутуу утга:')
print(df_model.isna().sum()[df_model.isna().sum() > 0])

In [ ]:
df_model.to_csv(OUT_PATH, index=False)
print(f'Хадгаллаа → {OUT_PATH}')
print(f'Хэмжээ: {len(df_model)} мөр × {len(df_model.columns)} багана')

## Дүгнэлт

Цэвэр өгөгдлийг `data/listings_clean.csv` файлд хадгаллаа. Дараагийн алхамд:

- **`03_eda.ipynb`** — өгөгдлийн дотоод бүтцийг судлах (корреляц, scatter, boxplot)
- **`04_models.ipynb`** — Шугаман регресс, Lasso, Шийдвэрийн мод, Random Forest харьцуулах